In [1]:
import sys
sys.path.append('../ExtraFiles/')
sys.path.append('../../RunData/') # Gets access to ReadInspiralData: Needed for restarts
sys.path.append('../')
import pandas as pd
import numpy as np
import time
import os
import mesa_reader as mr

# Needed for MESA integration
import MESAGen      as mg
import EnergyInjections as ei
import EnergyHelper as mh
import InlistHelper as ih
import utility      as rd    # Needed for Restart Runs [Need to Work]
import CEIO         as ceio
import CEphysics    as cephys
import CEParameters as cp
import CEconstants  as const
import scipy.integrate as spi

In [2]:
#context     = mg.pyMesa_context('../pyMesaCache/')

In [3]:
def eom(pv,t,ps):
    a,th,m1,Om,runParameters = pv
    bind,rs,rhos,_ms,css,gams,Ebs,Ms,Rns,dMs,dRs,enus,Is,h0s = ps
    
    rMin   = runParameters.accretorRadi
    jetEff = runParameters.jetEff
    freeCd = runParameters.freeCd
    dt     = runParameters.dtType
    
    # Donor Star Values
    _rho = np.interp(a,rs,rhos)
    _m   = np.interp(a,rs[1:],_ms)
    _v   = np.sqrt(G*(m1+_m)/a)
    _P   = np.sqrt(4.*np.pi**2./(G*(m1+_m))*a**3.)
    _Ra  = cePhys.Racc(m1,_v)
    _cs  = np.interp(a,rs,css)
    _gam = np.interp(a,rs,gams)
    _Ma  = _v/_cs
    _E   = -0.5*G*m1*_m/a                             # Orbital Energy
    _Eb  = np.interp(a,rs,Ebs)                        # Binding Energy
    epGrad = cePhys.getEPGrad(rhos,rs,a,_Ra, _rho)     # Macleod Density Gradient
    
    # Accretor Values
    Omks  = np.sqrt(G*Ms/Rns**3.)                     # Keplerian Angular Velocity
    mTOV  = np.interp(m1, Ms+(Om/Omks)**2.*dMs, Ms)   # Interpolated from NS EOS.... Mass Adjustment?
    _R    = np.interp(mTOV,Ms,Rns)                    # Interpolated Radi from NS EOS
    _I    = np.interp(mTOV,Ms,Is)                     # Inertial?
    _dR   = np.interp(mTOV,Ms,dRs)                    # Difference in R
    dI_dM = np.interp(mTOV,Ms,np.gradient(Is,Ms))
    _fk   = 1./(2.*np.pi)*np.sqrt(G*mTOV/_R**3.)      # Keplerian Frequency
    _f    = Om/(2.*np.pi)                             # Frequency
    _eps  = _f/_fk
    _ep2  = _eps*_eps
    _Rep  = _R + _ep2*_dR
    Rs_Ra = _Rep/_Ra

    if(int(runParameters.dragFeedback) == 1):
        dmb_dt, _Fd = cePhys.getAcc2019Drag(m1, _rho, _v, m1/_m, _Ma, _gam, Rs_Ra)
        
    elif(int(runParameters.dragFeedback) == 2):
        dmb_dt, _Fd   = cePhys.getNS2015Drag(m1, _rho, _v, _Ra, epGrad)
        
    elif(int(runParameters.dragFeedback) == 3):
        _Fd    = cePhys.getOst1999Drag(m1, _Ma, _rho, _v, rmin, dt)
        dmb_dt = cePhys.Mdot_HL(m1, _rho, _v)
        
    elif(int(runParameters.dragFeedback) == 4):
        I      = cePhys.getCoulombLog(_Ma, _R, rMin)
        _Fd    = cePhys.getBronnerDrag(m1, _cs, _Ma,_rho, freeCd, _R, _v, I)
        dmb_dt = cePhys.Mdot_HL(m1, _rho, _v)
        
    elif(int(runParameters.dragFeedback) == 5):
        dmb_dt, _Fd   = cePhys.getHybirdDrag(m1, _Ma, _rho, _v, a, _Ra, _cs, rMin, dt, epGrad)
    else:
        print('Error: No drag description chose...')
    
    # Calculating Jet Feedback...
    dE_dt  = - _Fd * _v
    dEjet = cePhys.getJetfeedback(m1, dmb_dt, rMin, jetEff)
    
    if(bind):
        _enu,_h0 = np.interp(mTOV,Ms,enus),np.interp(mTOV,Ms,h0s)
        _hep     = _ep2*_h0
        _phi     = np.sqrt(_enu*(1.+2.*_hep))
        dm_dt    = _phi*dmb_dt
    # end if

    dm_da = 4.*np.pi*_rho*a*a
    #da_dt = a*(dm_dt/m1 - dE_dt/_E)/(1.- a/_m*dm_da)  # Other way to calculate inspiral decay?
    da_dt = a*(dE_dt/np.abs(_E))
    dOm_dt = dm_dt/_I*(np.sqrt(G*m1*_Rep) - Om*dI_dM)
    dth_dt = 2.*np.pi/_P
    
    dz_dt  = [da_dt, dth_dt, dm_dt, dOm_dt, dmb_dt, dE_dt, dEjet]
    vars1  = [_m, _v, _P, _rho, _Ra, _cs, _gam, _Ma, _E, _Eb, _Fd, epGrad]
    return dz_dt, vars1

In [4]:
def integrate(context, pv, pv2, ps, runParameters):
    
    m1,a,t1,Om,th,m20,Edrag0,Ejet0,dmbn0   = pv
    v0,P0,rho0,Ra0,cs0,gam0,Ma0,Eorb0,Eb0  = pv2
    bind,Ms,Rns,dMs,dRs,enus,Is,h0s   = ps
    
    tn, an, thn, mn1, mn2, Omn, dmbn, Edl, Ejetl  = [[] for _ in range(9)]   # 1D data
    vn, pn, rhon, Ran, csn, gamn, Man, Eorbn, Ebn, Fdn, Epn = [[] for _ in range(11)]  # Donor Values at which accretor is located

    tn.append(t1/yr); an.append(a/Rsun); thn.append(th); mn1.append(m1/Msun); mn2.append(m20/Msun); 
    Omn.append(Om); dmbn.append(dmbn0); Edl.append(Edrag0); Ejetl.append(Ejet0);
    vn.append(v0); pn.append(P0), rhon.append(rho0); Ran.append(Ra0/Rsun); csn.append(cs0); 
    gamn.append(gam0); Man.append(Ma0); Eorbn.append(Eorb0); Ebn.append(Eb0); Fdn.append(0)
    Epn.append(0)
    
    runFlag, mesaFlag = True, True
    wallclock  = count = savenum = ckpoint = 0 # Run time, iter count, save iter number, chkpt counter
    nmax = int(10e9)                     # Max iteration
    
    #addRotation(info.inlistFile,False,0.95*P0)

    while(runFlag):
        itertime = time.time()
        Rs,rhos,_ms,css,gams,Ebs = mg.getMESAData(runParameters.profile)
        
        if (runParameters.dtType != 0): # Setting timestep
            dt = runParameters.dtType
        else:
            _m = np.interp(a,Rs[1:],_ms)
            _P = np.sqrt(4.*np.pi**2./(G*(m+_m))*a**3.)
            dt = np.max([0.005*_P,1.e-4*yr])
        # endif
        
        t2 = t1+dt
        pv = [a,th,m1,Om,runParameters]
        ps = [bind,Rs,rhos,_ms,css,gams,Ebs,M,Rn,dM,dR,enu,I,h0]
        
        varm1, varm2               = eom(pv,t2,ps)
        da,dth,dm,dOm,dmb,dE,dEjet = varm1
        _m, _v, _P, _rho, _Ra, _cs, _gam, _Ma, _Eorb, _Eb, _Fd, epGrad = varm2
        Edrag, Ejet = dE*dt, dEjet*dt    # Drag, Jet Energies
        m1,a,th,Om  = m1+dt*dm, a+dt*da, th+dt*dth, Om+dt*dOm
        
        # =================== Termination Criteria ======================
        if(a < runParameters.Rends): 
            runFlag = False; print('Reached final seperation')
        elif(count > nmax): 
            runFlag = False; print('Maxed iterations')
        elif(m1 > 1.9*Msun):
            runFlag = False; print('mass cut-off: M > 1.9Msun', m/Msun)
        elif(mesaFlag == False):
            runFlag = mesaFlag; print('Mesa stopped running')
            continue
            
        mg.includeCompactPotential(runParameters.inlistFile,runParameters.nsFlag, m1, a)
        mg.includeMassLoss(runParameters.inlistFile, runParameters.windFlag)
        
        if (runParameters.mesaRun == True):
            #if (count % 10 == 0):    # To match Brommmer... every 10 note iteration, call Mesa...
            # Part 1: Modify mod file to deposit energy...
            #  -------------------------- Drag Feedback ---------------------
            if (runParameters.includeDrag == True):
                upperDrag, lowerDrag = a + runParameters.freeCh*_Ra, a - runParameters.freeCh*_Ra
                dragZones            = mh.getAccretionZones(runParameters.modFile, upperDrag, lowerDrag, runParameters.zoneLimit)
                dragEnergyZ, dragRa  = mh.getEnergyDistribution(runParameters.modFile, dragZones, abs(Edrag), runParameters.dragKernel)
                zonaldragData        = pd.DataFrame({'Zone Index': dragZones, 'Zone Energy': dragEnergyZ , 'Radius': dragRa})
                ceIO.save1dData(zonaldragData, 'dragdonorZones_' + str(savenum), 'Zones Index, Zone Energy, Radius', runParameters)
                ei.callMesaPart1(runParameters, dragZones, dragEnergyZ, context)      # IMPORTANT: only edits mod file to add IE

            #  -------------------------- Jet Feedback ---------------------
            if (runParameters.includeJet == True):
                upperJet, lowerJet   = 1.0*Rs[-1], a
                jetZones             = mh.getAccretionZones(runParameters.modFile, upperJet, lowerJet, runParameters.zoneLimit)
                jetEnergyZ, jetRa    = mh.getEnergyDistribution(runParameters.modFile, dragZones, abs(Ejet), runParameters.jetKernel)
                zonaljetData         = pd.DataFrame({'Zone Index': jetZones, 'Zone Energy': jetEnergyZ, 'Radius': jetRa})
                ceIO.save1dData(zonaljetData, 'jetdonorZones_' + str(savenum), 'Zones Index, Zone Energy', runParameters)
                ei.callMesaPart1(runParameters, jetZones, jetEnergyZ, context)      # IMPORTANT: only edits mod file to add IE
            # ------------------------ Calling MESA -------------------------
            mesaFlag = ei.callMesaPart2(runParameters, dt/yr)
        # end Mesa If
        
        # Print Relevant Data...
        if(count % runParameters.saveFreq == 0):
            table_data = [
                ['Iteration:               ', count], 
                ['SaveNum:                 ', savenum], 
                ['Time                 [yr]', t2/yr],
                ['Donor Mass         [Msun]', _m/Msun],
                ['Accretor Mas       [Msun]', m1/Msun],
                ['Donors Max Radius  [Rsun]', max(Rs)/Rsun],
                ['Accretor Position  [Rsun]', a/Rsun],
                ]
            print("==========================================================")
            #for row in table_data:
            #    print("{: >10} | {: >10}".format(*row))
            
            tn.append(t2/yr); an.append(a/Rsun); thn.append(th); mn1.append(m1/Msun); mn2.append(_m/Msun);
            Omn.append(Om); dmbn.append(dmb); Edl.append(abs(Edrag)); Ejetl.append(Ejet); 
            vn.append(_v); pn.append(_P), rhon.append(_rho); Ran.append(_Ra/Rsun); csn.append(_cs);
            gamn.append(_gam); Man.append(_Ma); Eorbn.append(_Eorb); Ebn.append(_Eb); Fdn.append(_Fd)
            Epn.append(epGrad)
    
            tnData, anData, m1Data, m2Data, dmbnData = np.array(tn[:-1]), np.array(an[:-1]), np.array(mn1[:-1]), np.array(mn2[:-1]), np.array(dmbn[:-1])
            OmnData, EdragData, EjetData             = np.array(Omn[:-1]), np.array(Edl[:-1]), np.array(Ejetl[:-1])
            vnData, pnData, rhonData, RanData        = np.array(vn[:-1]), np.array(pn[:-1]), np.array(rhon[:-1]), np.array(Ran[:-1])
            csnData, gamnData, ManData, EorbnData    = np.array(csn[:-1]), np.array(gamn[:-1]), np.array(Man[:-1]), np.array(Eorbn[:-1])
            EbnData, FdnData, EpnData                = np.array(Ebn[:-1]), np.array(Fdn[:-1]), np.array(Epn[:-1])
            thnData                                  = np.array(thn[:-1])
            
            inspiralData = pd.DataFrame({'ts': tnData, 'rs': anData, 'm1s': m1Data, 'ms': m2Data, 'dmbs': dmbnData, 'Omn': OmnData, 'Edrag': EdragData, 'EJet': EjetData,
                                         'vn': vnData, 'ps': pnData, 'rhon': rhonData, 'Ran': RanData, 'csn': csnData, 'gamn': gamnData, 'Man': ManData, 'Eorbn': EorbnData,
                                         'Ebn': EbnData, 'Fdn': FdnData, 'Epn': EpnData, 'thn': thnData
                                        })
            inspiralNames = ['Time [yr]', 'sep [Rsun]', 'M1 [Msun]', 'M2 [Msun]', 'MdotM2 [g/s]', 'omn', 'Edrag [erg]', 'Ejet [erg]', 'V', 'P', 'Density', 'Racc [Rsun]', 'Csound', 'Gamma', 
                            'Mach', 'Eorbit', 'Ebind', 'Fd', 'EpGrad', 'thn']
            
            inspiralNames = ', '.join(inspiralNames)
            ceIO.save1dData(inspiralData, 'ceMain', inspiralNames, runParameters)       # Saves txt file of data
            ceIO.saveInspiralFile(runParameters.profile, runParameters.pathDonor, savenum, '_')    # Save Mesa profile
            print('Wall Clock Time: {} [secs]'.format(wallclock))
            print('\n')
            savenum += 1
            
        # Saving Checkpoint files of mod and profile for restarts...
        if(ckpoint < runParameters.ckptMax):
            ceIO.saveInspiralFile(runParameters.profile, runParameters.pathDonor, ckpoint, '_chkpt_')
            ceIO.saveInspiralFile(runParameters.modFile, runParameters.pathDonor, ckpoint, '_chkpt_')
            ckpoint += 1
        elif(ckpoint > runParameters.ckptMax):
            ckpoint = 0
            continue
            
        count += 1
        t1     = t2
        #turnOffOmega(info.inlistFile)
        wallclock += time.time()-itertime
        
    print('Run Finished: Saved Last Mod & Profile')
    ih.saveInspiralFile(runParameters.profile, runParameters.pathDonor, restart, '_restartPt_')   # Restart File
    ih.saveInspiralFile(runParameters.modFile, runParameters.pathDonor, restart, '_restartPt_')   # Restart File
# End of Run ...

In [5]:
context     = mg.pyMesa_context('../pyMesaCache/')

Loaded Modules for eos, const, math, chem...


In [6]:
ih.copyFile(runParameters.inspiralPath + '/InspiralRuntimeParameters', 
            runParameters.pathAccretor, 
            'InspiralRuntimeParameters.txt')
ih.copyFile(runParameters.inspiralPath + '/inlist_project',
            runParameters.pathAccretor, 
            'inlist_project.txt')

NameError: name 'runParameters' is not defined

In [ ]:
integrate(context, pv, pv2, ps, runParameters)